In [1]:
# Configuration initiale
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import udf
import time
from datetime import datetime, timedelta

# SparkSession avec configuration optimisée
spark = SparkSession.builder \
    .appName("PySpark_Avance") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

print("⚡ Configuration avancée PySpark initialisée")
print(f"🔥 Version: {spark.version}")
print(f"⚙️  Partitions par défaut: {spark.sparkContext.defaultParallelism}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/21 08:19:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


⚡ Configuration avancée PySpark initialisée
🔥 Version: 3.5.0
⚙️  Partitions par défaut: 24


## 1. Dataset complexe pour exercices avancés

In [2]:
# Génération d'un dataset plus complexe - Données transactionnelles
np.random.seed(42)

n_transactions = 50000
n_customers = 5000
n_products = 1000

# Simulation réaliste de données e-commerce
customers = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'customer_segment': np.random.choice(['Premium', 'Regular', 'Budget'], n_customers, p=[0.2, 0.5, 0.3]),
    'registration_date': pd.date_range('2020-01-01', '2023-12-31', periods=n_customers),
    'city': np.random.choice(['Paris', 'Lyon', 'Marseille', 'Toulouse', 'Nice'], n_customers)
})

products = pd.DataFrame({
    'product_id': range(1, n_products + 1),
    'category': np.random.choice(['Electronics', 'Clothing', 'Books', 'Home', 'Sports'], n_products),
    'price': np.round(np.random.uniform(10, 500, n_products), 2),
    'brand': np.random.choice(['BrandA', 'BrandB', 'BrandC', 'BrandD'], n_products)
})

transactions = pd.DataFrame({
    'transaction_id': range(1, n_transactions + 1),
    'customer_id': np.random.randint(1, n_customers + 1, n_transactions),
    'product_id': np.random.randint(1, n_products + 1, n_transactions),
    'quantity': np.random.randint(1, 5, n_transactions),
    'transaction_date': pd.date_range('2024-01-01', periods=n_transactions, freq='min'),
    'channel': np.random.choice(['online', 'store', 'mobile'], n_transactions, p=[0.6, 0.25, 0.15])
})

# Conversion en DataFrames Spark
df_customers = spark.createDataFrame(customers)
df_products = spark.createDataFrame(products)
df_transactions = spark.createDataFrame(transactions)

print(f"📊 Customers: {df_customers.count():,}")
print(f"📦 Products: {df_products.count():,}")
print(f"💳 Transactions: {df_transactions.count():,}")

# Cache pour performance
df_customers.cache()
df_products.cache()
df_transactions.cache()

print("💾 DataFrames mis en cache")

/tmp/ipykernel_6524/1701012213.py:28: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  'transaction_date': pd.date_range('2024-01-01', periods=n_transactions, freq='T'),
                                                                                

📊 Customers: 5,000
📦 Products: 1,000
💳 Transactions: 50,000
💾 DataFrames mis en cache


## 2. Window Functions - Concepts avancés

In [3]:
# Jointure des données pour analyse
df_full = df_transactions \
    .join(df_customers, 'customer_id') \
    .join(df_products, 'product_id') \
    .withColumn('total_amount', col('quantity') * col('price'))

print("🔗 Données consolidées")
df_full.printSchema()

# Cache du DataFrame principal
df_full.cache()
print(f"📊 Total lignes consolidées: {df_full.count():,}")

🔗 Données consolidées
root
 |-- product_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- transaction_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- channel: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- registration_date: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- brand: string (nullable = true)
 |-- total_amount: double (nullable = true)

📊 Total lignes consolidées: 50,000


In [4]:
# Window Functions - Analyses temporelles
print("📈 Window Functions - Analyses avancées")

# 1. Ranking des clients par montant total
window_customer_total = Window.partitionBy('customer_segment').orderBy(col('total_amount').desc())

df_customer_ranking = df_full.withColumn(
    'rank_in_segment',
    dense_rank().over(window_customer_total)
).withColumn(
    'percentile_in_segment',
    percent_rank().over(window_customer_total)
)

# Top 3 clients par segment
print("🏆 Top 3 clients par segment:")
df_customer_ranking.filter(col('rank_in_segment') <= 3) \
    .select('customer_id', 'customer_segment', 'total_amount', 'rank_in_segment', 'percentile_in_segment') \
    .orderBy('customer_segment', 'rank_in_segment') \
    .show()

📈 Window Functions - Analyses avancées
🏆 Top 3 clients par segment:
+-----------+----------------+------------+---------------+---------------------+
|customer_id|customer_segment|total_amount|rank_in_segment|percentile_in_segment|
+-----------+----------------+------------+---------------+---------------------+
|        752|          Budget|     1998.76|              1|                  0.0|
|       1383|          Budget|     1998.76|              1|                  0.0|
|        537|          Budget|     1996.64|              2|  0.15384615384615385|
|       2233|          Budget|     1996.64|              2|  0.15384615384615385|
|       3292|          Budget|     1996.64|              2|  0.15384615384615385|
|       3609|          Budget|     1996.64|              2|  0.15384615384615385|
|       3891|          Budget|     1996.64|              2|  0.15384615384615385|
|        990|          Budget|     1994.56|              3|   0.5384615384615384|
|       1573|          Budget|

In [5]:
# 2. Analyses temporelles avec LAG/LEAD
print("📅 Analyses temporelles avec LAG/LEAD")

# Fenêtre temporelle par client
window_customer_time = Window.partitionBy('customer_id').orderBy('transaction_date')

df_temporal = df_full.withColumn(
    'previous_purchase_amount',
    lag('total_amount', 1).over(window_customer_time)
).withColumn(
    'next_purchase_amount',
    lead('total_amount', 1).over(window_customer_time)
).withColumn(
    'running_total',
    sum('total_amount').over(window_customer_time.rowsBetween(Window.unboundedPreceding, Window.currentRow))
).withColumn(
    'purchase_number',
    row_number().over(window_customer_time)
)

# Exemple pour un client
print("👤 Historique d'un client (ID 1):")
df_temporal.filter(col('customer_id') == 1) \
    .select('transaction_date', 'total_amount', 'previous_purchase_amount', 'running_total', 'purchase_number') \
    .orderBy('transaction_date') \
    .show(10)

📅 Analyses temporelles avec LAG/LEAD
👤 Historique d'un client (ID 1):
+-------------------+-----------------+------------------------+-------------+---------------+
|   transaction_date|     total_amount|previous_purchase_amount|running_total|purchase_number|
+-------------------+-----------------+------------------------+-------------+---------------+
|2024-01-01 21:03:00|           325.84|                    NULL|       325.84|              1|
|2024-01-04 05:07:00|           697.68|                  325.84|      1023.52|              2|
|2024-01-08 01:47:00|            681.8|                  697.68|      1705.32|              3|
|2024-01-09 14:01:00|           195.54|                   681.8|      1900.86|              4|
|2024-01-13 08:47:00|           169.36|                  195.54|      2070.22|              5|
|2024-01-13 12:21:00|660.9000000000001|                  169.36|      2731.12|              6|
|2024-01-14 06:02:00|           164.02|       660.9000000000001|      2895.

## 3. UDF (User Defined Functions) Avancées

In [6]:
# Définition d'UDF personnalisées
print("🔧 User Defined Functions (UDF)")

# 1. UDF pour classification des clients
@udf(returnType=StringType())
def classify_customer_value(total_spent, num_purchases):
    """Classifier la valeur du client selon ses achats."""
    if total_spent > 2000 and num_purchases > 10:
        return "High Value"
    elif total_spent > 1000 and num_purchases > 5:
        return "Medium Value"
    elif total_spent > 300:
        return "Low Value"
    else:
        return "New Customer"

# 2. UDF pour score de fidélité
@udf(returnType=FloatType())
def loyalty_score(days_since_registration, num_purchases, avg_amount):
    """Calculer un score de fidélité client."""
    if days_since_registration == 0:
        return 0.0
    
    frequency_score = min(num_purchases / (days_since_registration / 30), 1.0)  # Achats par mois
    amount_score = min(avg_amount / 100, 1.0)  # Score basé sur montant moyen
    
    return (frequency_score * 0.6 + amount_score * 0.4) * 100

print("✅ UDF définies")

🔧 User Defined Functions (UDF)
✅ UDF définies


In [7]:
# Application des UDF
print("🎯 Application des UDF")

# Agrégation par client d'abord
df_customer_stats = df_full.groupBy('customer_id', 'customer_segment', 'registration_date', 'city') \
    .agg(
        sum('total_amount').alias('total_spent'),
        count('transaction_id').alias('num_purchases'),
        avg('total_amount').alias('avg_amount'),
        min('transaction_date').alias('first_purchase'),
        max('transaction_date').alias('last_purchase')
    ).withColumn(
        'days_since_registration',
        datediff(current_date(), col('registration_date'))
    )

# Application des UDF
df_enriched_customers = df_customer_stats \
    .withColumn(
        'customer_value_class',
        classify_customer_value(col('total_spent'), col('num_purchases'))
    ).withColumn(
        'loyalty_score',
        loyalty_score(col('days_since_registration'), col('num_purchases'), col('avg_amount'))
    )

print("📊 Statistiques clients enrichies:")
df_enriched_customers.show(10)

# Distribution des classes de clients
print("\n📈 Distribution des classes de clients:")
df_enriched_customers.groupBy('customer_value_class') \
    .agg(
        count('customer_id').alias('count'),
        avg('loyalty_score').alias('avg_loyalty_score')
    ).orderBy(col('avg_loyalty_score').desc()).show()

🎯 Application des UDF
📊 Statistiques clients enrichies:


25/11/21 08:19:31 ERROR Executor: Exception in task 0.0 in stage 26.0 (TID 318)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/tmp/ipykernel_6524/3819731898.py", line 24, in loyalty_score
  File "/workspace/venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/sql/utils.py", line 174, in wrapped
    return f(*args, **kwargs)
TypeError: min() takes 1 positional argument but 2 were given

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.sc

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/tmp/ipykernel_6524/3819731898.py", line 24, in loyalty_score
  File "/workspace/venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/sql/utils.py", line 174, in wrapped
    return f(*args, **kwargs)
TypeError: min() takes 1 positional argument but 2 were given


## 4. Optimisation et Performance Tuning

In [ ]:
# Stratégies de Cache et Persistence
print("💾 Stratégies de Cache et Persistence")

# 1. Différents niveaux de stockage
from pyspark import StorageLevel

# Cache mémoire seule
df_memory_only = df_full.cache()  # Équivalent à MEMORY_ONLY

# Cache mémoire + disque
df_memory_disk = df_enriched_customers.persist(StorageLevel.MEMORY_AND_DISK)

# Cache sérialisé (économise la mémoire)
df_serialized = df_customer_stats.persist(StorageLevel.MEMORY_ONLY_SER)

print("📊 Informations de cache:")
print(f"Partitions df_full: {df_full.rdd.getNumPartitions()}")
print(f"Partitions df_enriched_customers: {df_enriched_customers.rdd.getNumPartitions()}")

# Force l'exécution pour mettre en cache
count_full = df_memory_only.count()
count_enriched = df_memory_disk.count()
count_stats = df_serialized.count()

print(f"✅ Cache activé: {count_full}, {count_enriched}, {count_stats} lignes")

In [ ]:
# 2. Partitioning intelligent
print("🎯 Partitioning et optimisation")

# Repartitioning par colonne fréquemment utilisée
df_partitioned = df_full.repartition(8, 'customer_segment')  # 8 partitions par segment

# Coalesce pour réduire le nombre de partitions
df_coalesced = df_enriched_customers.coalesce(4)

print(f"Partitions originales df_full: {df_full.rdd.getNumPartitions()}")
print(f"Partitions après repartition: {df_partitioned.rdd.getNumPartitions()}")
print(f"Partitions après coalesce: {df_coalesced.rdd.getNumPartitions()}")

# Test de performance avec partitioning
start_time = time.time()
result_normal = df_full.filter(col('customer_segment') == 'Premium').count()
time_normal = time.time() - start_time

start_time = time.time()
result_partitioned = df_partitioned.filter(col('customer_segment') == 'Premium').count()
time_partitioned = time.time() - start_time

print(f"⏱️  Sans partitioning: {time_normal:.4f}s")
print(f"⏱️  Avec partitioning: {time_partitioned:.4f}s")
print(f"🚀 Amélioration: {((time_normal - time_partitioned) / time_normal * 100):.1f}%")

## 5. Analyses complexes avec SQL

In [ ]:
# Enregistrement des vues temporaires pour SQL
df_full.createOrReplaceTempView("transactions_full")
df_enriched_customers.createOrReplaceTempView("customers_enriched")

print("📝 Vues temporaires créées pour Spark SQL")

In [ ]:
# Requête SQL complexe - Cohort Analysis
cohort_query = """
WITH customer_months AS (
  SELECT 
    customer_id,
    DATE_TRUNC('month', registration_date) as cohort_month,
    DATE_TRUNC('month', transaction_date) as transaction_month,
    SUM(total_amount) as monthly_revenue
  FROM transactions_full
  GROUP BY customer_id, cohort_month, transaction_month
),
cohort_data AS (
  SELECT 
    cohort_month,
    DATEDIFF(transaction_month, cohort_month) / 30 as month_number,
    COUNT(DISTINCT customer_id) as customers,
    SUM(monthly_revenue) as revenue
  FROM customer_months
  GROUP BY cohort_month, month_number
)
SELECT 
  cohort_month,
  month_number,
  customers,
  revenue,
  ROUND(revenue / customers, 2) as revenue_per_customer
FROM cohort_data
WHERE month_number BETWEEN 0 AND 6
ORDER BY cohort_month, month_number
"""

print("📊 Cohort Analysis:")
cohort_analysis = spark.sql(cohort_query)
cohort_analysis.show(20)

In [ ]:
# Analyse RFM (Recency, Frequency, Monetary) avec SQL
rfm_query = """
WITH rfm_base AS (
  SELECT 
    customer_id,
    DATEDIFF(CURRENT_DATE(), MAX(transaction_date)) as recency,
    COUNT(transaction_id) as frequency,
    SUM(total_amount) as monetary
  FROM transactions_full
  GROUP BY customer_id
),
rfm_scores AS (
  SELECT 
    customer_id,
    recency,
    frequency,
    monetary,
    CASE 
      WHEN recency <= 30 THEN 5
      WHEN recency <= 60 THEN 4
      WHEN recency <= 90 THEN 3
      WHEN recency <= 120 THEN 2
      ELSE 1
    END as r_score,
    CASE 
      WHEN frequency >= 10 THEN 5
      WHEN frequency >= 7 THEN 4
      WHEN frequency >= 5 THEN 3
      WHEN frequency >= 3 THEN 2
      ELSE 1
    END as f_score,
    CASE 
      WHEN monetary >= 1000 THEN 5
      WHEN monetary >= 500 THEN 4
      WHEN monetary >= 200 THEN 3
      WHEN monetary >= 100 THEN 2
      ELSE 1
    END as m_score
  FROM rfm_base
)
SELECT 
  customer_id,
  recency,
  frequency,
  monetary,
  r_score,
  f_score,
  m_score,
  (r_score + f_score + m_score) as rfm_total_score,
  CASE 
    WHEN (r_score + f_score + m_score) >= 12 THEN 'Champions'
    WHEN (r_score + f_score + m_score) >= 9 THEN 'Loyal Customers'
    WHEN (r_score + f_score + m_score) >= 6 THEN 'Potential Loyalists'
    ELSE 'At Risk'
  END as customer_segment_rfm
FROM rfm_scores
ORDER BY rfm_total_score DESC
"""

print("🎯 Analyse RFM:")
rfm_analysis = spark.sql(rfm_query)
rfm_analysis.show(15)

# Distribution des segments RFM
print("\n📈 Distribution segments RFM:")
rfm_analysis.groupBy('customer_segment_rfm') \
    .agg(
        count('customer_id').alias('count'),
        avg('rfm_total_score').alias('avg_score'),
        avg('monetary').alias('avg_monetary')
    ).orderBy(col('avg_score').desc()).show()

## 6. Performance Monitoring et Debugging

In [ ]:
# Monitoring des performances
print("📊 Monitoring des performances")

# Informations sur le SparkContext
sc = spark.sparkContext
print(f"🖥️  Application ID: {sc.applicationId}")
print(f"⚙️  Cores totaux: {sc.defaultParallelism}")
print(f"🌐 Master: {sc.master}")
print(f"📍 Spark UI: http://localhost:4040")

# Informations sur les DataFrames en cache
print("\n💾 DataFrames en cache:")
for rdd in sc._jvm.org.apache.spark.storage.StorageLevel().toString():
    print(f"Cache info: {rdd}")

# Statistiques des requêtes
print("\n⚡ Performance de la dernière requête RFM:")
rfm_explain = rfm_analysis.explain()
print("Plan d'exécution disponible dans Spark UI")

In [ ]:
# Benchmark comparatif - Opérations complexes
print("🏁 Benchmark - Opérations complexes")

# Test 1: Agrégation complexe avec cache
start_time = time.time()
complex_agg_cached = df_full.groupBy('customer_segment', 'category', 'channel') \
    .agg(
        sum('total_amount').alias('total_revenue'),
        avg('total_amount').alias('avg_order'),
        count('transaction_id').alias('transaction_count'),
        countDistinct('customer_id').alias('unique_customers')
    ).cache()

result_cached = complex_agg_cached.collect()
time_cached = time.time() - start_time

# Test 2: Même opération sans cache
df_full.unpersist()  # Retirer du cache temporairement

start_time = time.time()
complex_agg_no_cache = df_full.groupBy('customer_segment', 'category', 'channel') \
    .agg(
        sum('total_amount').alias('total_revenue'),
        avg('total_amount').alias('avg_order'),
        count('transaction_id').alias('transaction_count'),
        countDistinct('customer_id').alias('unique_customers')
    )

result_no_cache = complex_agg_no_cache.collect()
time_no_cache = time.time() - start_time

# Remettre en cache
df_full.cache()

print(f"⏱️  Avec cache: {time_cached:.4f}s")
print(f"⏱️  Sans cache: {time_no_cache:.4f}s")
print(f"🚀 Amélioration cache: {((time_no_cache - time_cached) / time_no_cache * 100):.1f}%")
print(f"📊 Résultats: {len(result_cached)} lignes")

## 7. Patterns de migration Pandas → PySpark

In [ ]:
# Démonstration des patterns de migration courants
print("🔄 Patterns de migration Pandas → PySpark")

# Exemple de code Pandas typique à migrer
def pandas_analysis_example():
    """Exemple d'analyse Pandas à migrer vers PySpark."""
    # Simulation d'un dataset Pandas
    df_pandas = df_full.toPandas()
    
    print("🐼 Version Pandas:")
    start_time = time.time()
    
    # Analyse Pandas
    result_pandas = df_pandas.groupby(['customer_segment', 'category']).agg({
        'total_amount': ['sum', 'mean', 'std'],
        'quantity': 'sum',
        'customer_id': 'nunique'
    })
    
    # Aplatir les colonnes multi-index
    result_pandas.columns = ['_'.join(col).strip() for col in result_pandas.columns.values]
    result_pandas = result_pandas.reset_index()
    
    pandas_time = time.time() - start_time
    print(f"   ⏱️ Temps: {pandas_time:.4f}s")
    print(f"   📊 Shape: {result_pandas.shape}")
    
    return result_pandas, pandas_time

def pyspark_analysis_equivalent():
    """Version PySpark équivalente."""
    print("\n⚡ Version PySpark:")
    start_time = time.time()
    
    # Analyse PySpark
    result_spark = df_full.groupBy('customer_segment', 'category').agg(
        sum('total_amount').alias('total_amount_sum'),
        avg('total_amount').alias('total_amount_mean'),
        stddev('total_amount').alias('total_amount_std'),
        sum('quantity').alias('quantity_sum'),
        countDistinct('customer_id').alias('customer_id_nunique')
    ).orderBy('customer_segment', 'category')
    
    result_collected = result_spark.collect()
    spark_time = time.time() - start_time
    print(f"   ⏱️ Temps: {spark_time:.4f}s")
    print(f"   📊 Lignes: {len(result_collected)}")
    
    return result_spark, spark_time

# Exécution des deux versions
pandas_result, pandas_time = pandas_analysis_example()
spark_result, spark_time = pyspark_analysis_equivalent()

print(f"\n🏁 Comparaison finale:")
print(f"   📈 Speedup: {pandas_time / spark_time:.2f}x")
print(f"   💾 Mémoire: PySpark distribue automatiquement")

# Affichage du résultat PySpark
spark_result.show(10)

## 8. Exercices avancés

### Exercice 1: Analyse de panier d'achat
Trouvez les produits fréquemment achetés ensemble

In [ ]:
# TODO: Votre code ici
# Indice: groupBy customer_id et transaction_date, collect_list des produits
# Puis analyser les co-occurrences

### Exercice 2: Détection d'anomalies
Identifiez les transactions inhabituelles avec des window functions

In [ ]:
# TODO: Votre code ici
# Indice: utilisez mean() et stddev() dans une window function
# Transactions > mean + 2*stddev = anomalies

### Exercice 3: UDF pour scoring personnalisé
Créez une UDF complexe pour calculer un score de recommandation

In [ ]:
# TODO: Votre code ici
# Indice: UDF qui combine comportement d'achat, préférences produit, et récence

## 🎯 Points clés de ce notebook

### ⚡ Window Functions
- Puissantes pour analyses temporelles et ranking
- `partitionBy()` définit les groupes
- `orderBy()` définit l'ordre dans chaque groupe
- `rowsBetween()` pour fenêtres mobiles

### 🔧 UDF
- Permettent la logique métier personnalisée
- Attention aux performances (sérialisation Python)
- Privilégier les fonctions intégrées quand possible
- Typage explicite requis

### 💾 Optimisation
- Cache stratégique pour DataFrames réutilisés
- Partitioning intelligent selon les patterns d'accès
- Monitoring avec Spark UI
- Coalesce pour réduire les petites partitions

### 🔄 Migration
- API similaire mais pas identique
- Lazy evaluation change le paradigme
- SQL souvent plus expressif pour requêtes complexes
- Performance supérieure à partir d'un certain volume

In [ ]:
# Nettoyage final
print("🧹 Nettoyage des caches...")

# Libérer les caches
df_customers.unpersist()
df_products.unpersist()
df_transactions.unpersist()
df_full.unpersist()
df_memory_disk.unpersist()
df_serialized.unpersist()

print("✅ Caches libérés")

# Fermer la SparkSession
spark.stop()
print("🛑 SparkSession fermée")
print("\n🎉 Notebook avancé terminé ! Félicitations ! 🚀")